In [1]:
# model
from rich import print as rprint
from dotenv import load_dotenv
load_dotenv(override=True)
from langchain.chat_models import init_chat_model
model = init_chat_model(
    model="deepseek-v4-flash", # 模型名称
    extra_body={
        "thinking": {"type": "disabled"}
    }
)

## langchain集成mcp

MCP Client 与 MCP Server之间有两种通信协议 stdio 和 http <br>
1、stdio的通信方式，就是把这个MCP服务的脚本下载到本地，然后作为一个子进程运行（本地环境必须支持npx、uvx命令）<br>
2、http则是通过 TCP/IP 网络协议，发送标准的 HTTP 请求。<br>
核心区别一句话概括：stdio 是本地进程内通信（与单Agent进程绑定），HTTP 是跨网络远程通信（支持多Agent共享）

In [2]:
import asyncio, os
from langchain.agents import create_agent
from langchain_core.messages import HumanMessage
from langchain_mcp_adapters.client import MultiServerMCPClient

BAIDU_MAP_API_KEY = os.environ.get("BAIDU_MAP_API_KEY")

async def main():
    client = MultiServerMCPClient(
        {
            # "time": {
            #     "transport": "stdio",
            #     "command": "uvx",
            #     "args": ["mcp-server-time"]
            # },
            "baidu_map": {
                "transport": "streamable_http",
                "url": "https://mcp.map.baidu.com/mcp?ak=" + BAIDU_MAP_API_KEY
            }
        }
    )

    tools = await client.get_tools()

    agent = create_agent(
        tools=tools,
        model=model,
    )

    response = await agent.ainvoke({
        "messages": [("user", "请告诉我苏州的经纬度")]
    })

    print(response["messages"][-1].content)

if __name__ == "__main__":
    asyncio.run(main())


ImportError: cannot import name 'RequestContext' from 'mcp.shared.context' (D:\develop\project\agent-learning-notes\.venv\Lib\site-packages\mcp\shared\context.py)